# Spec S4b.2 — HyDE (Hypothetical Document Embeddings)

Interactive verification of the HyDE retrieval service.

**Flow**: User query → LLM generates hypothetical passage → Embed passage → KNN search → Real results

**Dependencies**: S4.4 (Hybrid Search), S5.1 (LLM Client)

In [1]:
import sys
sys.path.insert(0, "../..")

from src.config import HyDESettings, Settings

settings = Settings()
assert hasattr(settings, "hyde"), "HyDESettings not found in root Settings"
assert settings.hyde.enabled is True
assert settings.hyde.temperature == 0.3
assert settings.hyde.max_tokens == 300
assert settings.hyde.timeout == 30
print(f"HyDE enabled: {settings.hyde.enabled}")
print(f"Temperature: {settings.hyde.temperature}")
print(f"Max tokens: {settings.hyde.max_tokens}")
print(f"Timeout: {settings.hyde.timeout}s")
print("\n✓ HyDESettings configured correctly")

HyDE enabled: True
Temperature: 0.3
Max tokens: 300
Timeout: 30s

✓ HyDESettings configured correctly


## HyDEService — Unit Test with Mocks

Verify the full HyDE pipeline using mocked LLM, embeddings, and OpenSearch.

In [2]:
from unittest.mock import AsyncMock, MagicMock

from src.services.retrieval.hyde import HyDEService, HyDEResult
from src.services.llm.provider import LLMResponse, UsageMetadata

# Mock dependencies
mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=LLMResponse(
    text="Transformer architectures use self-attention to process sequences in parallel, "
         "replacing recurrence entirely. Multi-head attention enables the model to attend "
         "to different representation subspaces simultaneously.",
    model="gemini-2.0-flash",
    provider="gemini",
    usage=UsageMetadata(prompt_tokens=50, completion_tokens=60, total_tokens=110),
))

mock_embeddings = AsyncMock()
mock_embeddings.embed_query = AsyncMock(return_value=[0.1] * 1024)

mock_opensearch = MagicMock()
mock_opensearch.search_chunks_vectors = MagicMock(return_value={
    "total": 1,
    "hits": [{
        "arxiv_id": "1706.03762",
        "title": "Attention Is All You Need",
        "authors": ["Vaswani", "Shazeer"],
        "abstract": "The dominant sequence transduction models...",
        "pdf_url": "https://arxiv.org/pdf/1706.03762",
        "chunk_text": "We propose a new simple network architecture...",
        "chunk_id": "1706.03762_chunk_0",
        "section_title": "Introduction",
        "score": 0.95,
        "highlights": {},
    }],
})

hyde_settings = HyDESettings()
service = HyDEService(
    settings=hyde_settings,
    llm_provider=mock_llm,
    embeddings_client=mock_embeddings,
    opensearch_client=mock_opensearch,
)

# Test full flow (use await directly — Jupyter already has a running event loop)
result = await service.retrieve_with_hyde("What are transformers in NLP?")

assert isinstance(result, HyDEResult)
assert len(result.hypothetical_document) > 0
assert len(result.query_embedding) == 1024
assert len(result.results) == 1
assert result.results[0].arxiv_id == "1706.03762"

print(f"Hypothetical doc: {result.hypothetical_document[:80]}...")
print(f"Embedding dims: {len(result.query_embedding)}")
print(f"Results: {len(result.results)}")
print(f"Top hit: {result.results[0].title} (score={result.results[0].score})")
print("\n✓ HyDE full pipeline works correctly")

Hypothetical doc: Transformer architectures use self-attention to process sequences in parallel, r...
Embedding dims: 1024
Results: 1
Top hit: Attention Is All You Need (score=0.95)

✓ HyDE full pipeline works correctly


## Fallback Behavior

Verify that HyDE gracefully falls back to standard query embedding when the LLM fails.

In [3]:
# Test fallback: LLM failure should still return search results
mock_llm_fail = AsyncMock()
mock_llm_fail.generate = AsyncMock(side_effect=Exception("LLM unavailable"))

fallback_service = HyDEService(
    settings=hyde_settings,
    llm_provider=mock_llm_fail,
    embeddings_client=mock_embeddings,
    opensearch_client=mock_opensearch,
)

result = await fallback_service.retrieve_with_hyde("What are transformers?")

assert result.hypothetical_document == "What are transformers?"
assert len(result.results) == 1
print(f"Fallback doc: '{result.hypothetical_document}' (original query used)")
print(f"Results still returned: {len(result.results)}")
print("\n✓ Graceful fallback works correctly")

HyDE generation failed, falling back to original query
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/services/retrieval/hyde.py", line 69, in generate_hypothetical_document
    response = await self._llm.generate(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/unittest/mock.py", line 2282, in _execute_mock_call
    raise effect
Exception: LLM unavailable


Fallback doc: 'What are transformers?' (original query used)
Results still returned: 1

✓ Graceful fallback works correctly


## DI Registration

Verify HyDEDep is available in the dependency injection container.

In [4]:
from src.dependency import HyDEDep, get_hyde_service
from src.services.retrieval.hyde import HyDEService
from src.services.retrieval.factory import create_hyde_service

assert HyDEDep is not None
assert callable(get_hyde_service)
assert callable(create_hyde_service)

# Verify factory creates correct type
svc = create_hyde_service(
    settings=hyde_settings,
    llm_provider=mock_llm,
    embeddings_client=mock_embeddings,
    opensearch_client=mock_opensearch,
)
assert isinstance(svc, HyDEService)
print("HyDEDep registered in dependency.py")
print("create_hyde_service factory works")
print("\n✓ DI integration complete")

HyDEDep registered in dependency.py
create_hyde_service factory works

✓ DI integration complete
